# spatialpandas

## Example 1: a point array and its coordinates

In [ ]:
# Example 1: PointArray stores x/y pairs as an Arrow-backed pandas extension array
import pandas as pd
from spatialpandas import GeoSeries
from spatialpandas.geometry import PointArray

points = PointArray([[0, 1], [1, 2], None, [-1, -2], [7, 3]], dtype="float64")
series = GeoSeries(points)
assert len(series) == 5 and series.isna().tolist() == [False, False, True, False, False]
assert list(points.x[[0, 1, 3, 4]]) == [0, 1, -1, 7] and list(points.y[[0, 1, 3, 4]]) == [1, 2, -2, 3]
assert points.total_bounds == (-1.0, -2.0, 7.0, 3.0)
print("Example 1:", series.dtype, "total bounds", points.total_bounds)

## Example 2: polygon areas, holes included

In [ ]:
# Example 2: PolygonArray takes flat x0,y0,x1,y1,... rings (outer first, then holes); area is computed by numba
from spatialpandas.geometry import PolygonArray

square = [[0, 0, 4, 0, 4, 4, 0, 4, 0, 0]]
with_hole = [[0, 0, 4, 0, 4, 4, 0, 4, 0, 0], [1, 1, 1, 2, 2, 2, 2, 1, 1, 1]]
polys = PolygonArray([square, with_hole])
assert list(polys.area) == [16.0, 15.0], list(polys.area)
assert list(polys.length) == [16.0, 20.0], list(polys.length)
print("Example 2: areas", list(polys.area), "perimeters", list(polys.length))

## Example 3: a GeoDataFrame and a bounding-box selection

In [ ]:
# Example 3: cx[x0:x1, y0:y1] keeps the rows whose geometry intersects the box
from spatialpandas import GeoDataFrame
from spatialpandas.geometry import PointArray

gdf = GeoDataFrame({"geometry": PointArray([[0, 0], [1, 1], [2, 2], [5, 5], [9, 9]]), "v": [10, 11, 12, 13, 14]})
inside = gdf.cx[0.5:5.5, 0.5:5.5]
assert inside["v"].tolist() == [11, 12, 13], inside["v"].tolist()
assert gdf.geometry.name == "geometry"
print("Example 3:", len(inside), "of", len(gdf), "rows in the box:", inside["v"].tolist())

## Example 4: round trip with GeoPandas

In [ ]:
# Example 4: from_geopandas / to_geopandas convert both ways through shapely geometries
import geopandas
from shapely.geometry import Point, Polygon
from spatialpandas import GeoSeries

gs = geopandas.GeoSeries([Polygon([(0, 0), (2, 0), (2, 2), (0, 2)]), Polygon([(0, 0), (1, 0), (1, 1)])])
sp = GeoSeries(gs)
assert str(sp.dtype).startswith("polygon") and list(sp.area) == [4.0, 0.5]
back = sp.to_geopandas()
assert back.geom_equals(gs).all()
print("Example 4:", sp.dtype, "areas", list(sp.area), "round trip equal:", bool(back.geom_equals(gs).all()))

## Example 5: a Dask GeoDataFrame and a spatial join

In [ ]:
# Example 5: from_pandas gives a DaskGeoDataFrame; sjoin finds which polygon each point falls in
import dask.dataframe as dd
from spatialpandas import GeoDataFrame, sjoin
from spatialpandas.dask import DaskGeoDataFrame
from spatialpandas.geometry import PointArray, PolygonArray

pts = GeoDataFrame({"geometry": PointArray([[0.5, 0.5], [1.5, 0.5], [3, 3], [0.2, 0.8]]), "id": [1, 2, 3, 4]})
zones = GeoDataFrame({"geometry": PolygonArray([[[0, 0, 1, 0, 1, 1, 0, 1, 0, 0]], [[1, 0, 2, 0, 2, 1, 1, 1, 1, 0]]]),
                      "zone": ["A", "B"]})
dpts = dd.from_pandas(pts, npartitions=2)
assert isinstance(dpts, DaskGeoDataFrame)
joined = sjoin(dpts, zones).compute().sort_values("id")
assert joined[["id", "zone"]].values.tolist() == [[1, "A"], [2, "B"], [4, "A"]], joined
print("Example 5:", type(dpts).__name__, "->", joined[["id", "zone"]].values.tolist())